# 07 — Eval aggregators on test groups (Phase 2, шаги 12 + 13 + 14)

**Цель.** Посчитать NDCG@10/20 на `test_groups` (из `groups_split.pkl`) для 4 обучаемых агрегаторов + 3 тривиальных бейзлайнов (AVG/LM/MP). Посчитать bootstrap CI, срез по размеру группы, paired bootstrap для Audio-* vs ID-*, собрать финальную таблицу и LaTeX-фрагмент для ВКР.

Все 3 шага Phase 2 (12 — eval, 13 — analysis, 14 — тривиальные бейзлайны) объединены в этот ноутбук, чтобы (а) текст ВКР цитировал одну сквозную таблицу с одним bootstrap-протоколом, (б) не было ноутбука 08 с дублированием bootstrap-обвязки.

Контракт:
1. HF-download артефактов (идемпотентно)
2. `groups_split.pkl` → `test_samples`
3. Для каждого метода: `best.pt` → `predict_group_scores(test_samples)` → NDCG@10/20
4. Bootstrap CI 1000 resamples, **единая resample-сетка** для всех методов
5. Срез по размеру группы + heatmap
6. Paired bootstrap: AudioAGREE−AGREE, AudioAGREE−GroupIM, GroupCrossAttn−AGREE, GroupCrossAttn−GroupIM
7. Save → csv/npz в `artifacts/eval_results/`
8. Анализ: финальная таблица, forest plot, LaTeX-фрагмент

In [ ]:
# Colab bootstrap (раскомментировать в Colab):
# from google.colab import userdata
# token = userdata.get('git')
# !git clone -q -b models-1 https://$token@github.com/Vladislavbro/music-recommendations.git
# %cd music-recommendations
# !pip install -q datasets pyarrow numpy pandas torch huggingface_hub matplotlib

In [ ]:
import os, sys, json, pickle, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
print('project root:', PROJECT_ROOT)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# Подгружаем артефакты из HF-датасета. Локально (Mac) файлы уже есть — hf_hub_download
# просто вернёт путь из кэша. На Colab скачает в artifacts/.
from huggingface_hub import hf_hub_download

HF_REPO = 'Vladislavbro-500/music-recommendations'
HF_REPO_TYPE = 'dataset'
ARTIFACTS = PROJECT_ROOT / 'artifacts'
ARTIFACTS.mkdir(parents=True, exist_ok=True)

needed = [
    # Phase 1
    'gsasrec/item_id_to_idx.pkl',
    'user_scores_cache/scores.parquet',
    # audio
    'audio/embeddings.npy',
    'audio/user_profiles.npy',
    'audio/uid_to_row.pkl',
    'audio/user_audio_valid.npy',
    # groups split (зафиксирован в шаге 11)
    'aggregators/groups_split.pkl',
    # 4 чекпоинта
    'aggregators/agree/best.pt',            'aggregators/agree/config.json',
    'aggregators/groupim/best.pt',          'aggregators/groupim/config.json',
    'aggregators/audio_agree/best.pt',      'aggregators/audio_agree/config.json',
    'aggregators/group_cross_attn/best.pt', 'aggregators/group_cross_attn/config.json',
]
for rel in needed:
    dst = ARTIFACTS / rel
    if dst.exists():
        print(f'  [skip]  {rel}  ({dst.stat().st_size / 2**20:.1f} MB)')
        continue
    print(f'  [pull]  {rel} ...', flush=True)
    p = hf_hub_download(
        repo_id=HF_REPO,
        repo_type=HF_REPO_TYPE,
        filename=rel,
        local_dir=str(ARTIFACTS),
    )
    print(f'           -> {p}')

## 1. Data setup

Повторяем протокол Phase 1 / шага 11, чтобы test_df совпадал по timestamp-ам и item_remap с тем, на чём учился скорер и агрегаторы.

In [ ]:
from grouprec.data.yambda_loader import (
    load_yambda, filter_listens, filter_min_popularity, apply_item_remap,
)
from grouprec.data.splits import global_temporal_split, SplitConfig

HF_CACHE = os.environ.get('HF_DATASETS_CACHE', None)
raw = load_yambda('50m', cache_dir=HF_CACHE)['interactions']
df = filter_listens(raw)
df = filter_min_popularity(df, min_count=5)

with open(ARTIFACTS / 'gsasrec' / 'item_id_to_idx.pkl', 'rb') as f:
    item_id_to_idx = pickle.load(f)
df = apply_item_remap(df, item_id_to_idx)
assert df['item_idx'].isna().sum() == 0
df['item_idx'] = df['item_idx'].astype('int64')
n_items = max(item_id_to_idx.values())

train_df, val_df, test_df = global_temporal_split(df, SplitConfig())
print(f'events: train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}')
print(f'users:  train={train_df["uid"].nunique():,}  val={val_df["uid"].nunique():,}  test={test_df["uid"].nunique():,}')
print(f'n_items: {n_items:,}')

In [ ]:
from grouprec.eval.group_eval import topk_from_score_cache, test_targets_from_df
from grouprec.training.group_trainer import build_user_score_lookup, compute_pop_counts

scores_df = pd.read_parquet(ARTIFACTS / 'user_scores_cache' / 'scores.parquet')
print('scores rows:', len(scores_df), '| uids:', scores_df['uid'].nunique(),
      '| K per uid:', int(scores_df.groupby('uid').size().iloc[0]))

t0 = time.time()
user_topk = topk_from_score_cache(scores_df)
user_score_lookup = build_user_score_lookup(scores_df)
print(f'built user_topk + score_lookup in {time.time()-t0:.1f}s, n_users={len(user_topk):,}')

In [ ]:
item_audio = np.load(ARTIFACTS / 'audio' / 'embeddings.npy')
user_profiles = np.load(ARTIFACTS / 'audio' / 'user_profiles.npy')
with open(ARTIFACTS / 'audio' / 'uid_to_row.pkl', 'rb') as f:
    uid_to_row = pickle.load(f)

audio_valid_items = np.linalg.norm(item_audio, axis=1) > 0
print(f'item_audio: {item_audio.shape}, valid {int(audio_valid_items[1:].sum()):,}/{item_audio.shape[0]-1:,} '
      f'({100*audio_valid_items[1:].mean():.2f}%)')
print(f'user_profiles: {user_profiles.shape}, uid_to_row: {len(uid_to_row):,} entries')

In [ ]:
# Загружаем фиксированный split групп из шага 11. test_groups зафиксированы seed=42.
with open(ARTIFACTS / 'aggregators' / 'groups_split.pkl', 'rb') as f:
    gsplit = pickle.load(f)
test_groups = gsplit['test_groups']
print(f"group_seed={gsplit['group_seed']}, size_dist={gsplit['size_dist']}")
print(f"sizes: train={len(gsplit['train_groups'])}, val={len(gsplit['val_groups'])}, test={len(test_groups)}")

In [ ]:
# Сборка GroupSample на test. target = union(test listens членов) ∩ candidates.
# Контракт согласован с шагом 5 (group_eval.py).
from grouprec.eval.group_eval import build_group_samples

user_test_targets = test_targets_from_df(test_df)
print(f'users with test listens: {len(user_test_targets):,}')

test_samples, test_stats = build_group_samples(
    test_groups, user_topk, user_test_targets,
    ground_truth='union', drop_empty=True, drop_missing_member=True,
)
print('TEST stats:')
for k in ['n_input_groups', 'n_kept', 'n_dropped_missing_member',
          'n_dropped_empty_candidates', 'n_dropped_empty_targets',
          'candidate_size_mean', 'target_size_mean', 'by_size_counts']:
    print(f'  {k}: {test_stats[k]}')

## 2. Per-method group scores на test

Для каждой обучаемой модели: восстанавливаем архитектуру с дефолтами из шага 11, грузим `best.pt`, прогоняем `predict_group_scores(test_samples)`. Через `evaluate_aggregator_scores` получаем массивы per-sample NDCG@10/20 — они и пойдут в bootstrap.

Тривиальные бейзлайны (AVG/LM/MP, шаг 14) считаются здесь же одной функцией без обучения.

In [ ]:
from dataclasses import replace
from grouprec.training.group_trainer import GroupTrainConfig, GroupAggregatorTrainer
from grouprec.aggregators import IDBasedAGREE, GroupIM, AudioAGREE, GroupCrossAttention
from grouprec.eval.group_eval import evaluate_aggregator_scores

user_pool = sorted(user_topk.keys())
AGG_DIR = ARTIFACTS / 'aggregators'

# Дефолты конфига идентичны шагу 11 (только n_epochs/patience неважны — мы не учим).
EVAL_CFG = GroupTrainConfig(
    n_epochs=0, batch_size=64, eval_batch_size=128, lr=1e-3,
    weight_decay=0.0, n_neg_per_pos=4, eval_k=(10, 20),
    early_stop_patience=1, seed=42, device=DEVICE, log_every_steps=0,
)
# pop_counts здесь не нужны для eval, но Trainer требует — передаём заглушку
# (predict_group_scores их не использует).
pop_counts = compute_pop_counts(train_df, n_items=n_items, item_col='item_idx', smoothing=0.75)

METHODS = [
    {'name': 'AGREE',         'subdir': 'agree',
     'ctor': lambda: IDBasedAGREE(uid_list=user_pool, num_items=n_items, d_emb=32, d_att=32)},
    {'name': 'GroupIM',       'subdir': 'groupim',
     'ctor': lambda: GroupIM(uid_list=user_pool, num_items=n_items, d_emb=32, d_att=32)},
    {'name': 'AudioAGREE',    'subdir': 'audio_agree',
     'ctor': lambda: AudioAGREE(d_audio=128, d_att=64)},
    {'name': 'GroupCrossAttn','subdir': 'group_cross_attn',
     'ctor': lambda: GroupCrossAttention(d_audio=128, d_model=64, n_heads=4)},
]
for m in METHODS:
    p = AGG_DIR / m['subdir'] / 'best.pt'
    assert p.exists(), f'missing checkpoint: {p}'
print('all 4 checkpoints found')

In [ ]:
# Прогон 4 обучаемых методов: best.pt → predict_group_scores → NDCG per-sample.
results = {}              # name -> dict (point estimates + per_sample arrays)
group_scores_per_method = {}  # name -> list[np.ndarray[|C|]]

for m in METHODS:
    name = m['name']
    sub = m['subdir']
    cfg = replace(EVAL_CFG, out_dir=str(AGG_DIR / sub))
    model = m['ctor']()
    sd = torch.load(AGG_DIR / sub / 'best.pt', map_location=cfg.device, weights_only=False)
    model.load_state_dict(sd['aggregator_state'])
    trainer = GroupAggregatorTrainer(
        aggregator=model, cfg=cfg,
        user_score_lookup=user_score_lookup, pop_counts=pop_counts,
        item_audio=item_audio, user_profiles=user_profiles, uid_to_row=uid_to_row,
    )
    t0 = time.time()
    gs = trainer.predict_group_scores(test_samples)
    metrics = evaluate_aggregator_scores(test_samples, gs, k_list=list(EVAL_CFG.eval_k))
    print(f'{name:18s} | NDCG@10={metrics["NDCG@10"]:.4f}  NDCG@20={metrics["NDCG@20"]:.4f}  '
          f'n={metrics["n_samples"]}  ({time.time()-t0:.1f}s)')
    results[name] = metrics
    group_scores_per_method[name] = gs
    del model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

### 2.1 Тривиальные бейзлайны (шаг 14)

- **AVG**: `s_G(i) = mean_u s_{u,i}`  
- **LM** (Least Misery): `s_G(i) = min_u s_{u,i}`  
- **MP** (Most Pleasure): `s_G(i) = max_u s_{u,i}`

Здесь `s_{u,i}` = тот же кэш SASRec, что подаётся в обучаемые методы; для item ∉ top-K юзера ставится `fill=0.0` — идентично контракту `GroupTrainConfig.fill_score` (шаг 6). Это гарантирует, что бейзлайны и обучаемые методы работают с одним и тем же сырьём.

In [ ]:
from grouprec.training.group_trainer import lookup_per_user_scores

TRIVIAL_AGGS = {'AVG': np.mean, 'LM': np.min, 'MP': np.max}

def trivial_group_scores(samples, score_lookup, agg_fn, fill=0.0):
    out = []
    for s in samples:
        cands = s.candidates
        per_user = np.stack([
            lookup_per_user_scores(score_lookup, int(u), cands, fill=fill)
            for u in s.members
        ], axis=0)  # [G, C]
        out.append(agg_fn(per_user, axis=0).astype(np.float32))
    return out

for name, fn in TRIVIAL_AGGS.items():
    t0 = time.time()
    gs = trivial_group_scores(test_samples, user_score_lookup, fn, fill=EVAL_CFG.fill_score)
    metrics = evaluate_aggregator_scores(test_samples, gs, k_list=list(EVAL_CFG.eval_k))
    print(f'{name:18s} | NDCG@10={metrics["NDCG@10"]:.4f}  NDCG@20={metrics["NDCG@20"]:.4f}  '
          f'n={metrics["n_samples"]}  ({time.time()-t0:.1f}s)')
    results[name] = metrics
    group_scores_per_method[name] = gs

METHOD_ORDER = ['AGREE', 'GroupIM', 'AudioAGREE', 'GroupCrossAttn', 'AVG', 'LM', 'MP']
assert set(results) == set(METHOD_ORDER)

## 3. Bootstrap CI

1000 resamples, общая `rng(seed=42)`. Те же resample-индексы переиспользуются в paired bootstrap ниже — это удешевляет код и делает доверительные интервалы паиров согласованными с маржинальными.

In [ ]:
N_BOOT = 1000
BOOT_SEED = 42
KS = list(EVAL_CFG.eval_k)

n_samples = len(test_samples)
rng = np.random.default_rng(BOOT_SEED)
resample_idx = rng.integers(0, n_samples, size=(N_BOOT, n_samples))  # [B, n]

def bootstrap_ci(per_sample_vals, idx, alpha=0.05):
    means = per_sample_vals[idx].mean(axis=1)  # [B]
    lo = np.quantile(means, alpha / 2)
    hi = np.quantile(means, 1 - alpha / 2)
    return float(per_sample_vals.mean()), float(lo), float(hi), means

per_sample = {name: {k: results[name]['per_sample'][f'NDCG@{k}'].astype(np.float64) for k in KS}
              for name in METHOD_ORDER}
boot_means = {name: {} for name in METHOD_ORDER}

rows = []
for name in METHOD_ORDER:
    row = {'method': name}
    for k in KS:
        mean, lo, hi, means = bootstrap_ci(per_sample[name][k], resample_idx)
        boot_means[name][k] = means
        row[f'NDCG@{k}']      = mean
        row[f'NDCG@{k}_lo95'] = lo
        row[f'NDCG@{k}_hi95'] = hi
    rows.append(row)
summary_df = pd.DataFrame(rows).sort_values('NDCG@10', ascending=False).reset_index(drop=True)
print(summary_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

## 4. Срез по размеру группы

Та же bootstrap-обвязка, но per-size — на каждом size resample делается заново из подмножества индексов соответствующего размера. CI там более шумные (меньше n per size).

In [ ]:
sizes = np.array([s.size for s in test_samples], dtype=np.int64)
unique_sizes = sorted(np.unique(sizes).tolist())
print('group sizes on test:', {int(s): int((sizes == s).sum()) for s in unique_sizes})

size_rng = np.random.default_rng(BOOT_SEED + 1)
size_resample = {}
for s in unique_sizes:
    idx = np.where(sizes == s)[0]
    size_resample[s] = idx[size_rng.integers(0, idx.size, size=(N_BOOT, idx.size))]

size_rows = []
for name in METHOD_ORDER:
    row = {'method': name}
    for k in KS:
        vals = per_sample[name][k]
        for s in unique_sizes:
            idx = np.where(sizes == s)[0]
            mean = float(vals[idx].mean()) if idx.size > 0 else float('nan')
            means_b = vals[size_resample[s]].mean(axis=1)
            row[f'NDCG@{k}[s={s}]']      = mean
            row[f'NDCG@{k}[s={s}]_lo95'] = float(np.quantile(means_b, 0.025))
            row[f'NDCG@{k}[s={s}]_hi95'] = float(np.quantile(means_b, 0.975))
    size_rows.append(row)
size_df = pd.DataFrame(size_rows)
cols_means = ['method'] + [f'NDCG@{k}[s={s}]' for k in KS for s in unique_sizes]
print(size_df[cols_means].to_string(index=False, float_format=lambda x: f'{x:.4f}'))

In [ ]:
# Heatmap method × size для NDCG@10 (для финального графика в ВКР).
import matplotlib.pyplot as plt

heat = np.zeros((len(METHOD_ORDER), len(unique_sizes)), dtype=np.float64)
for i, name in enumerate(METHOD_ORDER):
    vals = per_sample[name][10]
    for j, s in enumerate(unique_sizes):
        idx = np.where(sizes == s)[0]
        heat[i, j] = vals[idx].mean() if idx.size > 0 else np.nan

fig, ax = plt.subplots(figsize=(6, 3.5))
im = ax.imshow(heat, aspect='auto', cmap='viridis')
ax.set_xticks(range(len(unique_sizes)), [f's={s}' for s in unique_sizes])
ax.set_yticks(range(len(METHOD_ORDER)), METHOD_ORDER)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        ax.text(j, i, f'{heat[i,j]:.3f}', ha='center', va='center',
                color='white' if heat[i,j] < heat.mean() else 'black', fontsize=9)
ax.set_title('NDCG@10 on test groups')
plt.colorbar(im, ax=ax)
plt.tight_layout()
FIG_DIR = PROJECT_ROOT / 'docs' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG_DIR / 'eval_heatmap_method_size.png', dpi=150)
plt.show()

## 5. Paired bootstrap: Audio-* vs ID-*

Для каждой пары (Audio-метод, ID-метод) на одной и той же resample-сетке считаем `Δ_b = mean(audio) − mean(id)` на bootstrap-выборке `b`. Возвращаем point estimate, 95% CI и one-sided p-value `Pr(Δ ≤ 0)` — нулевая гипотеза «audio не лучше ID».

In [ ]:
PAIRS = [
    ('AudioAGREE',     'AGREE'),
    ('AudioAGREE',     'GroupIM'),
    ('GroupCrossAttn', 'AGREE'),
    ('GroupCrossAttn', 'GroupIM'),
]

paired_rows = []
for a, b in PAIRS:
    for k in KS:
        diff_sample = per_sample[a][k] - per_sample[b][k]
        mean_diff = float(diff_sample.mean())
        diff_b = diff_sample[resample_idx].mean(axis=1)
        paired_rows.append({
            'audio_method': a,
            'id_method':    b,
            'K':            k,
            'delta_mean':   mean_diff,
            'delta_lo95':   float(np.quantile(diff_b, 0.025)),
            'delta_hi95':   float(np.quantile(diff_b, 0.975)),
            'p_one_sided':  float((diff_b <= 0).mean()),
        })
paired_df = pd.DataFrame(paired_rows)
print(paired_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

## 6. Save results

- `summary.csv` — point estimates + 95% CI, маржинально по всем группам.  
- `summary_by_size.csv` — то же с разбивкой по размеру группы.  
- `paired.csv` — paired bootstrap deltas + p-value.  
- `per_sample.npz` — массивы per-sample NDCG@10/20 для каждого метода (для шага 13 без перепрогона).

In [ ]:
EVAL_DIR = ARTIFACTS / 'eval_results'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

summary_df.to_csv(EVAL_DIR / 'summary.csv', index=False)
size_df.to_csv(EVAL_DIR / 'summary_by_size.csv', index=False)
paired_df.to_csv(EVAL_DIR / 'paired.csv', index=False)

persample_dump = {}
for name in METHOD_ORDER:
    for k in KS:
        persample_dump[f'{name}__NDCG@{k}'] = per_sample[name][k].astype(np.float32)
persample_dump['sizes'] = sizes.astype(np.int32)
persample_dump['resample_idx'] = resample_idx.astype(np.int32)
np.savez_compressed(EVAL_DIR / 'per_sample.npz', **persample_dump)

for p in sorted(EVAL_DIR.iterdir()):
    print(f'  {p.relative_to(PROJECT_ROOT)}  ({p.stat().st_size / 1024:.1f} KB)')

## 7. Анализ результатов (шаг 13)

Читаем из in-memory `summary_df` / `size_df` / `paired_df` / `per_sample` (всё уже посчитано выше). Эти ячейки готовят финальные артефакты для текста ВКР: таблицу с CI, forest plot, LaTeX-фрагмент.

### 7.1 Финальная таблица: `mean [lo95, hi95]`

Формат — компактный, без отдельных колонок под lo/hi. Готов к копированию в текст / в `to_latex`.

In [ ]:
def fmt_with_ci(mean, lo, hi, digits=4):
    return f'{mean:.{digits}f} [{lo:.{digits}f}, {hi:.{digits}f}]'

final_rows = []
for _, r in summary_df.iterrows():
    row = {'method': r['method']}
    for k in KS:
        row[f'NDCG@{k}'] = fmt_with_ci(r[f'NDCG@{k}'], r[f'NDCG@{k}_lo95'], r[f'NDCG@{k}_hi95'])
    final_rows.append(row)
final_df = pd.DataFrame(final_rows)
print(final_df.to_string(index=False))

### 7.2 Forest plot: NDCG@10 с 95% CI

Каждой строке таблицы соответствует точка + горизонтальная error bar. Сортировка — по mean убывая, чтобы лидер был сверху. Это естественный graph для ВКР.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ordered = summary_df.copy().reset_index(drop=True)  # уже отсортирован по NDCG@10
y = np.arange(len(ordered))[::-1]  # верх = лучший
means = ordered['NDCG@10'].values
lo = ordered['NDCG@10_lo95'].values
hi = ordered['NDCG@10_hi95'].values
errs = np.stack([means - lo, hi - means])
# Цвет — по семейству: audio синий, id оранжевый, trivial серый.
FAMILY = {'AudioAGREE': 'tab:blue', 'GroupCrossAttn': 'tab:blue',
          'AGREE': 'tab:orange', 'GroupIM': 'tab:orange',
          'AVG': 'gray', 'LM': 'gray', 'MP': 'gray'}
colors = [FAMILY[m] for m in ordered['method']]
ax.errorbar(means, y, xerr=errs, fmt='o', color='black', ecolor='gray',
            elinewidth=1.5, capsize=4, markersize=0)
for yi, (xi, mi, ci) in enumerate(zip(means, ordered['method'], colors)):
    ax.scatter([xi], [y[yi]], s=60, color=ci, zorder=3, label=mi)
ax.set_yticks(y, ordered['method'])
ax.set_xlabel('NDCG@10 (95% bootstrap CI)')
ax.set_title('Test groups: aggregator comparison')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'eval_forest_plot.png', dpi=150)
plt.show()

### 7.3 Audio gap по размеру группы

Гипотеза из шага 11: audio-преимущество растёт с размером группы. Здесь — точка + CI на каждом size для AudioAGREE vs AGREE.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
for name, color in [('AudioAGREE', 'tab:blue'), ('GroupCrossAttn', 'tab:cyan'),
                    ('AGREE', 'tab:orange'), ('GroupIM', 'tab:red'),
                    ('AVG', 'gray')]:
    means = []
    los, his = [], []
    for s in unique_sizes:
        vals = per_sample[name][10]
        idx = np.where(sizes == s)[0]
        m = vals[idx].mean()
        means_b = vals[size_resample[s]].mean(axis=1)
        means.append(m); los.append(np.quantile(means_b, 0.025)); his.append(np.quantile(means_b, 0.975))
    means = np.array(means); los = np.array(los); his = np.array(his)
    ax.errorbar(unique_sizes, means, yerr=[means-los, his-means], fmt='o-', color=color,
                label=name, capsize=3, markersize=5)
ax.set_xlabel('group size'); ax.set_ylabel('NDCG@10')
ax.set_xticks(unique_sizes)
ax.set_title('NDCG@10 vs group size (95% bootstrap CI)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'eval_ndcg_by_size.png', dpi=150)
plt.show()

### 7.4 Значимость пар Audio-* vs ID-*

Из `paired_df` собираем компактную таблицу с разницей и звёздочками: `*` p<0.05, `**` p<0.01, `***` p<0.001 (one-sided, H0: «audio не лучше»).

In [ ]:
def stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

paired_pretty = paired_df.copy()
paired_pretty['delta'] = paired_pretty.apply(
    lambda r: f"{r['delta_mean']:+.4f} [{r['delta_lo95']:+.4f}, {r['delta_hi95']:+.4f}]", axis=1)
paired_pretty['sig'] = paired_pretty['p_one_sided'].apply(stars)
paired_pretty['p'] = paired_pretty['p_one_sided'].apply(lambda x: f'{x:.3f}')
print(paired_pretty[['audio_method','id_method','K','delta','p','sig']].to_string(index=False))

### 7.5 LaTeX-фрагмент

Генерирует LaTeX-таблицу для прямой вставки в текст ВКР. Сохраняем в `artifacts/eval_results/summary_table.tex` — оттуда тянем через `\input{...}` если решим хранить рядом с текстом.

In [ ]:
latex_rows = []
for _, r in summary_df.iterrows():
    cells = [r['method']]
    for k in KS:
        cells.append(f"{r[f'NDCG@{k}']:.4f}")
        cells.append(f"[{r[f'NDCG@{k}_lo95']:.4f},\\,{r[f'NDCG@{k}_hi95']:.4f}]")
    latex_rows.append(' & '.join(cells) + ' \\\\')

header = ('\\begin{tabular}{l' + 'rr' * len(KS) + '}\n'
         '\\toprule\n'
         'Method' + ''.join([f' & NDCG@{k} & 95\\% CI' for k in KS]) + ' \\\\\n'
         '\\midrule')
footer = '\\bottomrule\n\\end{tabular}'
latex_str = '\n'.join([header] + latex_rows + [footer])

out_tex = EVAL_DIR / 'summary_table.tex'
out_tex.write_text(latex_str)
print(f'saved {out_tex}')
print('\n--- LaTeX ---\n')
print(latex_str)

## 8. Что дальше

- **Phase 3** (если время позволит): end-to-end fine-tune SASRec поверх обученных агрегаторов как ablation. Сравниваем frozen vs unfrozen в одной таблице.
- **Текст ВКР**: финальная таблица NDCG@10/20 — [artifacts/eval_results/summary_table.tex](../artifacts/eval_results/summary_table.tex), forest plot — [docs/figures/eval_forest_plot.png](../docs/figures/eval_forest_plot.png), heatmap — [docs/figures/eval_heatmap_method_size.png](../docs/figures/eval_heatmap_method_size.png), gap по размеру — [docs/figures/eval_ndcg_by_size.png](../docs/figures/eval_ndcg_by_size.png).
- Опционально — `hf upload Vladislavbro-500/music-recommendations artifacts/eval_results eval_results --type dataset`, чтобы сохранить результаты рядом с чекпоинтами (ячейка ниже).

In [ ]:
# Раскомментировать в Colab.
# from google.colab import userdata
# os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
# !hf upload Vladislavbro-500/music-recommendations \
#     artifacts/eval_results eval_results \
#     --type dataset \
#     --commit-message 'phase2 step12: test eval (NDCG + bootstrap + paired)'